# Energy Consumption Forecast

In [1]:
import os
import re
from typing import List, Tuple
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import plotly.express as px
from countries_dataframe_generation import get_dataframe, know_processing_unit
import numba
# import conda

import tensorflow as tf

conda_prefix = os.environ.get('CONDA_PREFIX', '')
os.environ['XLA_FLAGS'] = f'--xla_gpu_cuda_data_dir={conda_prefix}'

print("GPU Available: ", tf.config.list_physical_devices('GPU'))
if tf.config.list_physical_devices('GPU'):
    print("Using GPU")
    # Configure GPU memory growth
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(e)
else:
    print("Using CPU")


know_processing_unit()

country = "Brazil"
Nationality = "Brazilian"
df = get_dataframe(country)



GPU Available:  []
Using CPU
--- Processing Unit Information ---
CPU: Intel(R) Core(TM) Ultra 7 155U

--- GPU & Accelerator Status ---
ℹ️ TensorFlow is using CPU. No compatible GPU found by TensorFlow.
ℹ️ Numba: No CUDA-enabled GPU detected.

-----------------------------------


# Train / Test Split

In [ ]:
def train_test_split(df_full: pd.DataFrame, cutoff: pd.Timestamp, end_test: pd.Timestamp) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Splits df_full into df_train and df_test based on the timestamp column 'time'.
    All rows with time <= '2023-12-31 23:00:00' go to df_train,
    and the remaining go to df_test.

    Returns:
        df_train, df_test: Tuple of DataFrames
    """
    # Ensure 'time' is datetime type
    if not pd.api.types.is_datetime64_any_dtype(df_full['time']):
        df_full = df_full.loc[:, ~df_full.columns.duplicated(keep='first')]
        df_full = df_full.copy()
        df_full['time'] = pd.to_datetime(df_full['time'])

    
    df_train = df_full[df_full['time'] <= cutoff].reset_index(drop=True)
    df_test = df_full[df_full['time'] > cutoff].reset_index(drop=True)
    df_test = df_test[df_test['time'] <= end_test].reset_index(drop=True)

    return df_train, df_test

cutoff_train_test = pd.Timestamp('2023-12-31 23:00:00')
end_test = pd.Timestamp('2024-12-31 23:00:00')

df_train, df_test = train_test_split(df, cutoff_train_test, end_test)
print(df_train.tail())
print(df_test.head())
print(df_test.tail())
mode = None

# Pre-processing

## Calendar

In [ ]:
# Import the functions from your new calendar module
from calendar_utils import calendar_df, plot_train_test_features

# Add calendar features to the dataframes
df_train_cal = calendar_df(df_train.copy(), country, Nationality)
df_test_cal = calendar_df(df_test.copy(), country, Nationality)

# Print tails and heads to verify
print("--- Train DF with Calendar Features (Tail) ---")
print(df_train_cal.tail())
print("\n--- Test DF with Calendar Features (Head) ---")
print(df_test_cal.head())

# Plot the features
plot_train_test_features(df_train_cal, df_test_cal, Nationality)

# Set the mode to indicate which features have been applied
mode = "calendar"

## Fourier Signal

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import warnings
from DEOS import deos, stairway_to_heaven_signal, plot_fft_light

n_peaks = 40
trunc_factor = 2

# Call the deos function to get all FFT analysis results
f_plot, m_plot, peak_freqs, peak_mags, keep_freqs, keep_mags = deos(
    df_train=df_train,
    # trunc_factor=trunc_factor,
    n_peaks=n_peaks,
    th_hours=6.00,
    th_week=2.00,
    th_months=2.00,
    th_years=2.00
)

# Plot the FFT results
plot_fft_light(country, n_peaks, f_plot, m_plot, peak_freqs, peak_mags, keep_freqs, keep_mags)

df_train, df_test = stairway_to_heaven_signal(df_train = df_train, df_test = df_test,
                                        keep_freqs = keep_freqs,
                                        keep_mags = keep_mags)
print(df_train.tail())
print(df_test.head())
if mode is not None:
    print("You are mixing Fourier Ramps and calendar features")
    mode = "Fourier Ramps and calendar"
else:
    print("Applying Fourier Ramps")
    mode = "Fourier Ramps"


# Train, Test, and Compare Models

In [ ]:
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def plot_correlation_matrix(df_train: pd.DataFrame):
    # Select only relevant columns
    corr_features = [col for col in df_train.columns if col not in {'time', 'timestamp', 'value'}]
    df_corr = df_train[corr_features]

    # Compute Pearson correlation matrix
    corr_matrix = df_corr.corr(method='pearson')

    # Plotly heatmap
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Pearson<br>Correlation")
    ))

    fig.update_layout(
        title='Pearson Correlation Matrix (Features vs Value)',
        template='plotly_white',
        width=600,
        height=600
    )

    fig.show()

def plot_pca(df_train: pd.DataFrame):
    """
    Performs PCA and plots the results, including the feature contributions (loadings).
    """
    # 1. Select only numerical columns for PCA
    features_for_pca = [col for col in df_train.columns if pd.api.types.is_numeric_dtype(df_train[col])]
    df_pca_input = df_train[features_for_pca].dropna()
    print(f"✅ Performing PCA on the following {len(features_for_pca)} features: {features_for_pca}")

    color_values = df_pca_input.get('value', None)

    # 2. Scale the data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_pca_input)

    # 3. Apply PCA
    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(scaled_data)

    # --- START OF THE NEW CODE ---

    # 4. Show which original features contribute to PC1 and PC2
    loadings = pca.components_
    df_loadings = pd.DataFrame(
        loadings,
        columns=features_for_pca,
        index=['Principal Component 1', 'Principal Component 2']
    )
    
    print("\n--- Principal Component Loadings ---")
    print("This table shows the 'recipe' for each Principal Component.")
    print("A large positive or negative value means the feature is highly influential.")
    print(df_loadings.round(3))
    print("\n" + "="*50 + "\n")

    # 5. Visualize the loadings with a heatmap
    fig_loadings = go.Figure(data=go.Heatmap(
        z=df_loadings.values,
        x=df_loadings.columns,
        y=df_loadings.index,
        colorscale='RdBu', # Red for negative, Blue for positive contributions
        zmin=-1, zmax=1,
        text=np.around(df_loadings.values, 2),
        texttemplate="%{text}",
        colorbar=dict(title="Weight")
    ))
    fig_loadings.update_layout(
        title='Contribution of Original Features to Principal Components',
        template='plotly_white'
    )
    fig_loadings.show()

    # --- END OF THE NEW CODE ---

    # 6. Create the PCA scatter plot (your original plot)
    df_pca_results = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
    
    fig_scatter = go.Figure(data=go.Scatter(
        x=df_pca_results['PC1'],
        y=df_pca_results['PC2'],
        mode='markers',
        marker=dict(
            color=color_values,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Value")
        ),
        hovertemplate='<b>Value:</b> %{marker.color:.2f}<br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
    ))

    explained_variance = pca.explained_variance_ratio_
    fig_scatter.update_layout(
        title='Principal Component Analysis of Training Data',
        xaxis_title=f'Principal Component 1 ({explained_variance[0]:.1%})',
        yaxis_title=f'Principal Component 2 ({explained_variance[1]:.1%})',
        template='plotly_white',
        width=800,
        height=600
    )
    fig_scatter.show()

if mode is not None:
    print(f"Mode: {mode}")
    if mode == "calendar":
        df_train = df_train_cal
        df_test = df_test_cal

plot_correlation_matrix(df_train)
# plot_pca(df_train)

## One time 

### Random Forest

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import multiprocessing as mp
import plotly.graph_objects as go
import os

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

# === FEATURE & TARGET SETUP ===
exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]
X_tr = df_train[features].to_numpy()
X_te = df_test[features].to_numpy()
y_tr = df_train['value'].to_numpy().ravel()
y_te = df_test['value'].to_numpy().ravel()

# === HYPERPARAMETERS ===
n_features = len(features)  # Use all features
max_depth = 12
n_estimators = 100

# === MODEL TRAINING ===
model = RandomForestRegressor(
    max_depth=max_depth,
    n_estimators=n_estimators,
    bootstrap=True,
    oob_score=True,
    n_jobs=mp.cpu_count() // 2,
    warm_start=True
)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

# === METRICS ===
rmse_pct = 100 * np.sqrt(mean_squared_error(y_te, y_pred)) / np.mean(y_te)
mape = 100 * np.mean(np.abs((y_te - y_pred) / y_te))
print(f"Final Model: RMSE={rmse_pct:.2f}%, MAPE={mape:.2f}%")

# === PLOT ===
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_test['time'],
    y=y_te,
    name='Actual',
    line=dict(color='#002776')  # Brazilian blue flag color
))
fig.add_trace(go.Scatter(
    x=df_test['time'],
    y=y_pred,
    name='Predicted',
    line=dict(color='#009C3B')  # Brazilian green flag color
))
fig.update_layout(
    title='Actual vs Predicted Values Over Time',
    xaxis_title='Time',
    yaxis_title='Value',
    template='plotly_white'
)
fig.show()

### XGBoost

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import multiprocessing as mp
import plotly.graph_objects as go

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

# === FEATURE & TARGET SETUP ===
exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]
X_tr = df_train[features].to_numpy()
X_te = df_test[features].to_numpy()
y_tr = df_train['value'].to_numpy().ravel()
y_te = df_test['value'].to_numpy().ravel()

# === HYPERPARAMETERS ===
n_features = len(features)  # Use all features
max_depth = 20
n_estimators = 450

# === MODEL TRAINING ===
model = xgb.XGBRegressor(
    max_depth=max_depth,
    n_estimators=n_estimators,
    learning_rate=0.1,
    n_jobs=mp.cpu_count() // 2,
    verbosity=0
)
model.fit(X_tr, y_tr)

# === PREDICTIONS ===
y_pred_test = model.predict(X_te)
y_pred_train = model.predict(X_tr)

# === METRICS ===
test_rmse_pct = 100 * np.sqrt(mean_squared_error(y_te, y_pred_test)) / np.mean(y_te)
test_mape = 100 * np.mean(np.abs((y_te - y_pred_test) / y_te))

train_rmse_pct = 100 * np.sqrt(mean_squared_error(y_tr, y_pred_train)) / np.mean(y_tr)
train_mape = 100 * np.mean(np.abs((y_tr - y_pred_train) / y_tr))

print(f"Final Model Test Set: RMSE={test_rmse_pct:.2f}%, MAPE={test_mape:.2f}%")
print(f"Final Model Train Set: RMSE={train_rmse_pct}%, MAPE={train_mape}%")

# === PLOT ===
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_test['time'],
    y=y_te,
    name='Actual',
    line=dict(color='#002776')  # Brazilian blue flag color
))
fig.add_trace(go.Scatter(
    x=df_test['time'],
    y=y_pred_test,
    name='Predicted',
    line=dict(color='#009C3B')  # Brazilian green flag color
))
fig.update_layout(
    title='Actual vs Predicted Values Over Time (XGBoost)',
    xaxis_title='Time',
    yaxis_title='Value',
    template='plotly_white'
)
fig.show()


### LSTM

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

# === SEQUENCE CREATION ===
def create_sequences(X, y, n_steps):
    X_seq, y_seq = [], []
    for i in range(len(X) - n_steps):
        X_seq.append(X[i:i + n_steps])
        y_seq.append(y[i + n_steps])
    return np.array(X_seq), np.array(y_seq)

# === FEATURE & TARGET SETUP ===
n_steps = 1

exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]
X_tr = df_train[features].to_numpy()
X_te = df_test[features].to_numpy()
y_tr = df_train['value'].to_numpy().reshape(-1, 1)
y_te = df_test['value'].to_numpy().reshape(-1, 1)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_tr_scaled = scaler_X.fit_transform(X_tr)
X_te_scaled = scaler_X.transform(X_te)
y_tr_scaled = scaler_y.fit_transform(y_tr)
y_te_scaled = scaler_y.transform(y_te)

X_tr_seq, y_tr_seq = create_sequences(X_tr_scaled, y_tr_scaled, n_steps)
X_te_seq, y_te_seq = create_sequences(X_te_scaled, y_te_scaled, n_steps)

X_tr_seq = X_tr_seq.reshape((X_tr_seq.shape[0], n_steps, len(features)))
X_te_seq = X_te_seq.reshape((X_te_seq.shape[0], n_steps, len(features)))

# === LSTM CONFIGURATION ===
layers = [128, 64]  # You can modify this
epochs = 10
batch_size = 128
lr = 0.0001

# === MODEL TRAINING ===
model = Sequential()
model.add(Input(shape=(n_steps, len(features))))
for i, units in enumerate(layers):
    return_seq = i < len(layers) - 1
    model.add(LSTM(units, return_sequences=return_seq))
model.add(Dense(1))

model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
model.fit(X_tr_seq, y_tr_seq, epochs=epochs, batch_size=batch_size, verbose=1)

# === PREDICTIONS ===
y_pred_test = scaler_y.inverse_transform(model.predict(X_te_seq)).flatten()
y_pred_train = scaler_y.inverse_transform(model.predict(X_tr_seq)).flatten()

y_te_seq = scaler_y.inverse_transform(y_te_seq).flatten()
y_tr_seq = scaler_y.inverse_transform(y_tr_seq).flatten()


# === METRICS ===
test_rmse_pct = 100 * np.sqrt(mean_squared_error(y_te_seq, y_pred_test)) / np.mean(y_te_seq)
test_mape = 100 * np.mean(np.abs((y_te_seq - y_pred_test) / y_te_seq))
train_rmse_pct = 100 * np.sqrt(mean_squared_error(y_tr_seq, y_pred_train)) / np.mean(y_tr_seq)
train_mape = 100 * np.mean(np.abs((y_tr_seq - y_pred_train) / y_tr_seq))

print(f"Final Model Test Set: RMSE={test_rmse_pct:.2f}%, MAPE={test_mape:.2f}%")
print(f"Final Model Train Set: RMSE={train_rmse_pct:.2f}%, MAPE={train_mape:.2f}%")

# === PLOT ===
time_values = df_test['time'].to_numpy()[n_steps:]  # Adjust for sequence offset
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_values,
    y=y_te_seq,
    name='Actual',
    line=dict(color='#002776')
))
fig.add_trace(go.Scatter(
    x=time_values,
    y=y_pred_test,
    name='Predicted',
    line=dict(color='#009C3B')
))
fig.update_layout(
    title='Actual vs Predicted Values Over Time (LSTM)',
    xaxis_title='Time',
    yaxis_title='Value',
    template='plotly_white'
)
fig.show()


## Several times

In [ ]:
from many_times import verify_file, several_simulations


chosen_model = "Random_Forest"
results_path, plot_dir, csv_exists = verify_file(country, chosen_model)

if results_path == None:
    results_path = csv_exists

several_simulations(results_path, chosen_model, df_train, 
                       df_test)


chosen_model = "XGBoost"
results_path, plot_dir, csv_exists = verify_file(country, chosen_model)

if results_path == None:
    results_path = csv_exists

several_simulations(results_path, chosen_model, df_train, 
                       df_test)



chosen_model = "LSTM"
results_path, plot_dir, csv_exists = verify_file(country, chosen_model)

if results_path == None:
    results_path = csv_exists

several_simulations(results_path, chosen_model, df_train, 
                       df_test)

In [ ]:
from many_times import verify_file, plot_simulation_results

chosen_model = "XGBoost"
results_path, plot_dir, csv_exists = verify_file(country, chosen_model)
plot_simulation_results(results_path, chosen_model, country, Nationality, time_limit=20)

# Best Results Ran One Time (Fourier Ramps and Calendar) Comparison

In [ ]:
import pandas as pd
import ast
from many_times import model_forecast, verify_file
from DEOS import deos, stairway_to_heaven_signal
from calendar_utils import calendar_df

# --- Main Execution ---

# Define DEOS parameters from Cell 9
deos_params = {
    'n_peaks': 40, 
    # 'trunc_factor': 2, 
    'th_hours': 6.00, 
    'th_week': 2.00, 
    'th_months': 2.00, 
    'th_years': 2.00
}

# Map model names to their variable abbreviations
model_abbr_map = {
    "Random_Forest": "rf",
    "XGBoost": "xgb",
    "LSTM": "lstm"
}

# Loop through each model type to find and run its single best version
for model_name, model_abbr in model_abbr_map.items():
    try:
        print(f"\n{'='*20} Processing Best Model for: {model_name} {'='*20}")
        
        # 1. Load simulation results
        results_path, _, _ = verify_file(country, model_name)
        df_results = pd.read_csv(results_path)
        df_results.columns = df_results.columns.str.strip()
        
        # 2. Find the SINGLE best set of hyperparameters
        best_idx = df_results['rmse_percentual_test'].idxmin()
        best_hyperparams = df_results.loc[best_idx]
        best_mode = str(best_hyperparams['mode'])

        print(f"🏆 Best {model_name} Hyperparameters (Mode: '{best_mode}'):\n{best_hyperparams}")

        # 3. Prepare data ONLY with the best feature set
        df_train_orig, df_test_orig = train_test_split(df, cutoff_train_test, end_test) # Reset data for each model
        
        if best_mode == 'Calendar':
            print("Preparing data with Calendar features...")
            df_train_processed = calendar_df(df_train_orig.copy(), country, Nationality)
            df_test_processed = calendar_df(df_test_orig.copy(), country, Nationality)
        elif best_mode == 'Fourier Ramps':
            print("Preparing data with Fourier Ramps features...")
            _, _, _, _, freqs, mags = deos(df_train=df_train_orig.copy(), **deos_params)
            df_train_processed, df_test_processed = stairway_to_heaven_signal(
                df_train=df_train_orig.copy(), df_test=df_test_orig.copy(),
                keep_freqs=freqs, keep_mags=mags
            )
        else: # Handle mixed modes if they exist
            print("Preparing data with Mixed (Calendar + Fourier) features...")
            df_train_cal = calendar_df(df_train_orig.copy(), country, Nationality)
            df_test_cal = calendar_df(df_test_orig.copy(), country, Nationality)
            _, _, _, _, freqs, mags = deos(df_train=df_train_orig.copy(), **deos_params)
            df_train_processed, df_test_processed = stairway_to_heaven_signal(
                df_train=df_train_cal, df_test=df_test_cal,
                keep_freqs=freqs, keep_mags=mags
            )

        # 4. Build arguments and run the single best forecast
        features = [col for col in df_train_processed.columns if col not in ['time', 'value']]
        forecast_args = best_hyperparams.to_dict()
        forecast_args.update({
            'df_train': df_train_processed,
            'df_test': df_test_processed,
            'features': features,
        })

        # Clean and rename parameters
        if 'layers' in forecast_args and isinstance(forecast_args.get('layers'), str):
            forecast_args['layers'] = ast.literal_eval(forecast_args['layers'])
        if 'max_depth' in forecast_args: forecast_args['depth'] = forecast_args.pop('max_depth')
        if 'n_estimators' in forecast_args: forecast_args['ne'] = forecast_args.pop('n_estimators')
        if 'learning_rate' in forecast_args: forecast_args['lr'] = forecast_args.pop('learning_rate')
        if 'n_features' in forecast_args: forecast_args['nf'] = forecast_args.pop('n_features')
        if 'mode' in forecast_args: forecast_args['mod'] = forecast_args.pop('mode')

        # --- START OF FIX ---
        # 5. Dynamically create the correct variable name based on the mode
        if best_mode == 'Calendar':
            pred_var_name = f"y_pred_test_{model_abbr}_cal"
        else: # Default to '_fr' for 'Fourier Ramps' or mixed modes
            pred_var_name = f"y_pred_test_{model_abbr}_fr"
        # --- END OF FIX ---

        print(f"Running final forecast with {model_name}...")
        _, globals()[pred_var_name] = model_forecast(model_name, **forecast_args)
        print(f"✅ Generated prediction variable: {pred_var_name}")

    except Exception as e:
        print(f"❌ An error occurred during {model_name} forecast: {e}")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.figure_factory as ff
from typing import Dict

# --- Configuration Dictionaries ---
FLAG_COLORS = {
    'brazil': '#009C3B', 'sweden': '#006AA7', 'switzerland': '#DA291C',
    'spain': '#C60B1E', 'portugal': '#006600',
}

MODEL_COLORS = {
    'Random Forest (Fourier Ramps)': '#FFA500',
    'XGBoost (Fourier Ramps)': '#800080',
    'LSTM (Fourier Ramps)': '#008080',
    'Random Forest (Calendar)': '#FF4500',
    'XGBoost (Calendar)': '#9932CC',
    'LSTM (Calendar)': "#002FFF",
}

# --- Plotting Functions (re-included for completeness) ---
def plot_forecast_comparison(df_test: pd.DataFrame, country: str, **predictions: Dict[str, np.ndarray]):
    fig = go.Figure()
    actual_color = FLAG_COLORS.get(country.lower(), '#002776')
    fig.add_trace(go.Scatter(x=df_test['time'], y=df_test['value'], name='Actual Values', line=dict(color=actual_color, width=2.5)))
    for model_name, pred_values in predictions.items():
        time_values = df_test['time']
        if len(time_values) != len(pred_values): time_values = time_values.iloc[-len(pred_values):]
        fig.add_trace(go.Scatter(x=time_values, y=pred_values, name=f'{model_name} Forecast', line=dict(color=MODEL_COLORS.get(model_name, 'gray'), width=2, dash='dot')))
    # fig.update_layout(title=f'Forecast Comparison for {country.capitalize()}', xaxis_title='Time', yaxis_title='Energy Consumption (MWh/h)', template='plotly_white', legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01), 
    #     height=750,
    #     width=1800)
    
    x_min = df_test['time'][df_test['time']== pd.Timestamp('01-01-2024 00:00:00')].min()  # Change this to filter specific range
    x_max = df_test['time'][df_test['time'] == pd.Timestamp('12-31-2024 23:00:00')].max()   # Or use df_train['time'].max() for train only
    
    fig.update_layout(
        title=f'Forecast Comparison for {country.capitalize()}',
        xaxis_title="Time",
        yaxis_title="Energy Consumption (MWh/h)",
        title_font=dict(size=32),
        template='plotly_white',
        xaxis=dict(range=[x_min, x_max], title_font=dict(size=36), tickfont=dict(size=30)),
        # yaxis=dict(range=[55000, 120000], title_font=dict(size=36), tickfont=dict(size=30)), # Brazil
        # yaxis=dict(range=[55000, 120000], title_font=dict(size=36), tickfont=dict(size=30)), # Switzerland
        yaxis=dict(range=[5000, 20000], title_font=dict(size=36), tickfont=dict(size=30)), # Sweden
        # yaxis=dict(title_font=dict(size=36), tickfont=dict(size=30)), # standard
        legend=dict(
                font=dict(size=30),
                x=0.01,  # horizontal position (0-1)
                y=1,  # vertical position (0-1)
                bgcolor="rgba(255,255,255,0.8)",  # semi-transparent white background
                bordercolor="white",
                borderwidth=1
            ),
        height=1000,
        width=1400
    )
    fig.show()

    try:        
        fig.write_image(f"Results/{country}/Forecast_comparison.pdf", 
                        width=1000, height=800, scale=2)
        print(f"✅ PDF saved to Results/{country}/Forecast_comparison.pdf")
    except Exception as e:
        print(f"⚠️ Could not save PDF: {e}")
        print("Install kaleido with: pip install kaleido")

    print(f"Average energy consumption: {np.mean(df_test['value']):.2f} MWh/h")

def plot_error_distribution(df_test: pd.DataFrame, country: str, **predictions: Dict[str, np.ndarray]):
    error_data, group_labels, colors = [], [], []
    error_stats = []  # NEW: Store statistics for each model
    
    for model_name, pred_values in predictions.items():
        actual_values = df_test['value']
        if len(actual_values) != len(pred_values): actual_values = actual_values.iloc[-len(pred_values):]
        error = actual_values - pred_values
        error_data.append(error)
        group_labels.append(model_name)
        colors.append(MODEL_COLORS.get(model_name, 'gray'))
        
        # NEW: Calculate statistics
        mean_error = np.average(error)
        std_error = np.std(error)
        error_stats.append({
            'Model': model_name,
            'Mean Error': mean_error,
            'Std Dev': std_error
        })
    
    if not error_data: return
    
    # NEW: Print statistics table
    print("\n" + "="*70)
    print("ERROR STATISTICS FOR EACH MODEL")
    print("="*70)
    for stat in error_stats:
        print(f"\n{stat['Model']}:")
        print(f"  Mean Error:      {stat['Mean Error']:>12.4f} MWh/h\n  Mean Error:      {100*stat['Mean Error']/np.mean(df_test['value']):>12.4f} %")
        print(f"  Std Deviation:   {stat['Std Dev']:>12.4f} MWh/h\n  Std Deviation:   {100*stat['Std Dev']/np.mean(df_test['value']):>12.4f} %")
    print("\n" + "="*70 + "\n")
    
    fig = ff.create_distplot(error_data, group_labels, bin_size=200, show_rug=False, colors=colors)
    
    fig.update_layout(
        title=f'Forecast Error Distribution for {country.capitalize()}',
        xaxis_title="Error (Actual - Forecast)",
        yaxis_title="Density",
        title_font=dict(size=32),
        template='plotly_white',
        xaxis=dict(range=[-4200, 4200], title_font=dict(size=24), tickfont=dict(size=20)), # Sweden
        yaxis=dict(title_font=dict(size=24), tickfont=dict(size=20)),
        legend=dict(
                font=dict(size=20),
                x=0.01,  # horizontal position (0-1)
                y=1,  # vertical position (0-1)
                bgcolor="rgba(255,255,255,0.8)",  # semi-transparent white background
                bordercolor="white",
                borderwidth=1
            ),
        height=750,
        width=1800
    )
    fig.show()
    try:        
        fig.write_image(f"Results/{country}/Forecast_error_distribution.pdf", 
                        width=1000, height=800, scale=2)
        print(f"✅ PDF saved to Results/{country}/Forecast_error_distribution.pdf")
    except Exception as e:
        print(f"⚠️ Could not save PDF: {e}")
        print("Install kaleido with: pip install kaleido")

# --- Diagnostic and Plotting Logic ---

print("--- Checking for Available Model Predictions ---")

# Define all the predictions we expect from cell 23
expected_predictions = {
    'Random Forest (Fourier Ramps)': 'y_pred_test_rf_fr',
    'XGBoost (Fourier Ramps)': 'y_pred_test_xgb_fr',
    'LSTM (Fourier Ramps)': 'y_pred_test_lstm_fr',
    'Random Forest (Calendar)': 'y_pred_test_rf_cal',
    'XGBoost (Calendar)': 'y_pred_test_xgb_cal',
    'LSTM (Calendar)': 'y_pred_test_lstm_cal',
}

predictions_to_plot = {}

# Check for each expected variable and report its status
for model_label, var_name in expected_predictions.items():
    if var_name in locals():
        print(f"✅ Found: {model_label} (variable '{var_name}')")
        predictions_to_plot[model_label] = locals()[var_name]
    else:
        print(f"❌ Not Found: {model_label} (variable '{var_name}')")

print("\n" + "="*40 + "\n")

# Call the plotting functions only if we found at least one prediction
if predictions_to_plot:
    print("--- Generating Forecast Comparison Plot ---")
    plot_forecast_comparison(df_test, country, **predictions_to_plot)
    
    print("\n--- Generating Error Distribution Plot ---")
    plot_error_distribution(df_test, country, **predictions_to_plot)
else:
    print("No model predictions were found to plot. Please re-run cell 23 and check for errors.")


# Ancient Code

### Random Forest

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import multiprocessing as mp
import time
import os
import plotly.graph_objects as go
import plotly.io as pio

# === PLOTTING CONFIGURATION ===
pio.kaleido.scope.default_format = "png"
pio.kaleido.scope.default_width = 1200
pio.kaleido.scope.default_height = 600

# === CONFIGURATION ===
results_path = 'Results/BR/Random_Forest/rf_calendar.txt'
plot_dir = 'Results/BR/Random_Forest/Plots'
os.makedirs(plot_dir, exist_ok=True)

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df


# === FEATURE & TARGET SETUP ===
exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]
X_tr = df_train[features].to_numpy()
X_te = df_test[features].to_numpy()
y_tr = df_train['value'].to_numpy().ravel()
y_te = df_test['value'].to_numpy().ravel()

# === LOAD EXISTING RESULTS ===
if os.path.exists(results_path):
    df_existing = pd.read_csv(results_path, sep='\t', skipinitialspace=True)
    df_existing = clean_columns(df_existing)
    write_header = False
    write_mode = 'a'
else:
    df_existing = pd.DataFrame(columns=['n_features','max_depth','n_estimators','rmse_percentual','mape','more_than_10_min'])
    write_header = True
    write_mode = 'w'

existing_tuples = set(zip(df_existing['n_features'], df_existing['max_depth'], df_existing['n_estimators']))

# === HYPERPARAMETER GRID ===
max_depths = list(range(5, 21, 5))
n_estimators = list(range(50, 551, 100))
full_tuples = {
    (nf, d, ne)
    for nf in range(1, len(features)+1)
    for d in max_depths
    for ne in n_estimators
}
missing = full_tuples - existing_tuples
results_new = []

# === SIMULATION LOOP ===
for nf, depth, ne in sorted(missing):
    subset = features[:nf]
    X_sub_tr = df_train[subset].values
    X_sub_te = df_test[subset].values

    t0 = time.perf_counter()
    model = RandomForestRegressor(
        max_depth=depth,
        n_estimators=ne,
        bootstrap=True,
        n_jobs=mp.cpu_count() // 2,
        warm_start=True
    )
    model.fit(X_sub_tr, y_tr)
    elapsed = time.perf_counter() - t0

    y_pred = model.predict(X_sub_te)
    rmse_pct = 100 * np.sqrt(mean_squared_error(y_te, y_pred)) / np.mean(y_te)
    mape = 100 * np.mean(np.abs((y_te - y_pred) / y_te))

    filename = f"{plot_dir}/plot_nf{nf}_d{depth}_ne{ne}.png"
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_test['time'], y=y_pred, name='Predicted', line=dict(color='#DA1884')))
    fig.add_trace(go.Scatter(x=df_test['time'], y=y_te, name='Actual', line=dict(color='green', dash='dot')))
    fig.update_layout(title=f'Prediction vs Actual (nf={nf}, depth={depth}, est={ne})', xaxis_title='Time', yaxis_title='Value')
    # fig.write_image(filename)

    results_new.append({
        'n_features': nf,
        'max_depth': depth,
        'n_estimators': ne,
        'rmse_percentual': rmse_pct,
        'mape': mape,
        'more_than_10_min': elapsed 
    })

    print(f"Done: nf={nf}, depth={depth}, ne={ne}, RMSE={rmse_pct:.2f}%, MAPE={mape:.2f}%, time={elapsed/60:.1f}min")

# === SAVE RESULTS ===
df_new = pd.DataFrame(results_new)
df_new.to_csv(results_path, sep='\t', mode=write_mode, index=False, header=write_header)
print(f"{'Created' if write_header else 'Appended'} {len(df_new)} rows to {results_path}")

# === SUMMARY REPORT ===
df_all = pd.read_csv(results_path, sep='\t')
print("\n=== SUMMARY REPORT ===")
print("Columns found:", df_all.columns.tolist())
print("n_features values:", sorted(df_all['n_features'].unique()))
print("max_depth values:", sorted(df_all['max_depth'].unique()))

# === TEMPORAL ANALYSIS ===
results = pd.DataFrame({
    'Actual': y_te,
    'Predicted': y_pred,
    'Hour': df_test['time'].dt.hour,
    'DayOfWeek': df_test['time'].dt.dayofweek,
    'IsHoliday': df_test.get('is_holiday', pd.Series(False, index=df_test.index))
})

hourly_performance = results.groupby('Hour').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')
dow_performance = results.groupby('DayOfWeek').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')
holiday_performance = results.groupby('IsHoliday').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')

print("\nPerformance by Hour:")
print(hourly_performance)
print("\nPerformance by Day of Week:")
print(dow_performance)
print("\nPerformance on Holidays:")
print(holiday_performance)

# === SAVE PERFORMANCE PLOTS ===
def save_bar_plot(series, title, filename):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=series.index.astype(str), y=series.values, marker_color='#DA1884'))
    fig.update_layout(title=title, xaxis_title=series.name, yaxis_title='RMSE')
    fig.show()
    # fig.write_image(f"{plot_dir}/{filename}.png")

save_bar_plot(hourly_performance, 'RMSE by Hour of Day', 'hourly_rmse')
save_bar_plot(dow_performance, 'RMSE by Weekday', 'weekday_rmse')
save_bar_plot(holiday_performance, 'RMSE by Holiday', 'holiday_rmse')

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === 1) Load & clean ===
df = pd.read_csv(
    'Results/BR/Random_Forest/rf_evolution_filtered_integer.txt',
    sep='\t', skipinitialspace=True
)
df.columns = df.columns.str.strip()

# === 2) Prepare unique axis values ===
nf_feats = np.sort(df['n_features'].unique())
n_est    = np.sort(df['n_estimators'].unique())
max_d    = np.sort(df['max_depth'].unique())

# === 3) Pivot into grids ===
Z1_rmse = df.pivot_table(
    index='n_estimators', columns='n_features', values='rmse_percentual'
).loc[n_est, nf_feats].values
Z1_mape = df.pivot_table(
    index='n_estimators', columns='n_features', values='mape'
).loc[n_est, nf_feats].values

Z2_rmse = df.pivot_table(
    index='max_depth', columns='n_features', values='rmse_percentual'
).loc[max_d, nf_feats].values
Z2_mape = df.pivot_table(
    index='max_depth', columns='n_features', values='mape'
).loc[max_d, nf_feats].values

Z3_rmse = df.pivot_table(
    index='max_depth', columns='n_estimators', values='rmse_percentual'
).loc[max_d, n_est].values
Z3_mape = df.pivot_table(
    index='max_depth', columns='n_estimators', values='mape'
).loc[max_d, n_est].values

# === 4) Define Brazilian flag color palettes ===
gold_scale = [
    [0.0, '#FFDF00'], [0.5, '#FFEB66'], [1.0, '#FFF599']
]
green_scale = [
    [0.0, '#003D1A'], [0.5, '#007F3D'], [1.0, '#00BF7F']
]
blue_scale = [
    [0.0, '#001F3F'], [0.5, '#005F9E'], [1.0, '#009FFF']
]

# === 5) Build 3x2 subplot ===
fig = make_subplots(
    rows=3, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=[
        "RMSE (%) vs #Est × #Feats", "MAPE (%) vs #Est × #Feats",
        "RMSE (%) vs Depth × #Feats", "MAPE (%) vs Depth × #Feats",
        "RMSE (%) vs Depth × #Est",   "MAPE (%) vs Depth × #Est"
    ]
)

# === Helper to add a Mesh3D ===
def add_mesh(row, col, X, Y, Z, scale, name):
    scene_id = f"scene{(row-1)*2 + col}"
    fig.add_trace(
        go.Mesh3d(
            x=X.ravel(), y=Y.ravel(), z=Z.ravel(),
            intensity=Z.ravel(),
            colorscale=scale, opacity=0.7,
            showscale=False, name=name
        ),
        row=row, col=col
    )

# === 6) Add meshes ===
X1, Y1 = np.meshgrid(nf_feats, n_est)
add_mesh(1,1, X1, Y1, Z1_rmse, gold_scale, "RMSE")
add_mesh(1,2, X1, Y1, Z1_mape, gold_scale, "MAPE")

X2, Y2 = np.meshgrid(nf_feats, max_d)
add_mesh(2,1, X2, Y2, Z2_rmse, green_scale, "RMSE")
add_mesh(2,2, X2, Y2, Z2_mape, green_scale, "MAPE")

X3, Y3 = np.meshgrid(n_est, max_d)
add_mesh(3,1, X3, Y3, Z3_rmse, blue_scale, "RMSE")
add_mesh(3,2, X3, Y3, Z3_mape, blue_scale, "MAPE")

# === 7) Update layout ===
fig.update_layout(
    title="Random Forest Error Surfaces",
    height=1200, width=1400, showlegend=False
)

# === 8) Axis titles ===
for i, (xlab, ylab) in enumerate([
    ("#Features", "#Estimators"), ("#Features", "#Estimators"),
    ("#Features", "Max Depth"),   ("#Features", "Max Depth"),
    ("#Estimators", "Max Depth"), ("#Estimators", "Max Depth")
], start=1):
    scene = f"scene{i}"
    fig.update_scenes(
        xaxis_title=xlab,
        yaxis_title=ylab,
        zaxis_title="Error (%)"
    )

fig.show()


### XGBoost

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import multiprocessing as mp
import time
import os
import plotly.graph_objects as go
import plotly.io as pio

# === PLOTTING CONFIGURATION ===
pio.kaleido.scope.default_format = "png"
pio.kaleido.scope.default_width = 1200
pio.kaleido.scope.default_height = 600

# === CONFIGURATION ===
results_path = 'Results/BR/XGBoost/xgb_evolution_filtered_integer.txt'
plot_dir = 'Results/BR/XGBoost/Plots'
os.makedirs(plot_dir, exist_ok=True)

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

# === FEATURE & TARGET SETUP ===
exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]
X_tr = df_train[features].to_numpy()
X_te = df_test[features].to_numpy()
y_tr = df_train['value'].to_numpy().ravel()
y_te = df_test['value'].to_numpy().ravel()

# === LOAD EXISTING RESULTS ===
if os.path.exists(results_path):
    df_existing = pd.read_csv(results_path, sep='\t', skipinitialspace=True)
    df_existing = clean_columns(df_existing)
    write_header = False
    write_mode = 'a'
else:
    df_existing = pd.DataFrame(columns=['n_features','max_depth','n_estimators','rmse_percentual_train', 'rmse_percentual_test', 'mape_train', 'mape_test','more_than_10_min'])
    write_header = True
    write_mode = 'w'

existing_tuples = set(zip(df_existing['n_features'], df_existing['max_depth'], df_existing['n_estimators']))

# === HYPERPARAMETER GRID ===
max_depths = list(range(6, 31, 24))
n_estimators = list(range(101, 1001, 900))
full_tuples = {
    (nf, d, ne)
    for nf in range(1, len(features)+1)
    for d in max_depths
    for ne in n_estimators
}
missing = full_tuples - existing_tuples
results_new = []

# === SIMULATION LOOP ===
for nf, depth, ne in sorted(missing):
    subset = features[:nf]
    X_sub_tr = df_train[subset].values
    X_sub_te = df_test[subset].values

    t0 = time.perf_counter()
    model = xgb.XGBRegressor(
        max_depth=depth,
        n_estimators=ne,
        boostrap=True,
        oob_score=True,
        n_jobs=mp.cpu_count() // 2,
        tree_method="hist",
        verbosity=0
    )
    model.fit(X_sub_tr, y_tr)
    elapsed = time.perf_counter() - t0

    y_pred_test = model.predict(X_sub_te)
    y_pred_train = model.predict(X_sub_tr)

    # === METRICS ===
    test_rmse_pct = 100 * np.sqrt(mean_squared_error(y_te, y_pred_test)) / np.mean(y_te)
    test_mape = 100 * np.mean(np.abs((y_te - y_pred_test) / y_te))

    train_rmse_pct = 100 * np.sqrt(mean_squared_error(y_tr, y_pred_train)) / np.mean(y_tr)
    train_mape = 100 * np.mean(np.abs((y_tr - y_pred_train) / y_tr))

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_test['time'], y=y_pred, name='Predicted', line=dict(color='#DA1884')))
    fig.add_trace(go.Scatter(x=df_test['time'], y=y_te, name='Actual', line=dict(color='green', dash='dot')))
    fig.update_layout(title=f'Prediction vs Actual (nf={nf}, depth={depth}, est={ne})', xaxis_title='Time', yaxis_title='Value')


    results_new.append({
        'n_features': nf,
        'max_depth': depth,
        'n_estimators': ne,
        'rmse_percentual_train': train_rmse_pct,
        'rmse_percentual_test': test_rmse_pct,
        'mape_train': train_mape,
        'mape_test': test_mape,
        'more_than_10_min': elapsed / 60
    })

    print(f"Done: nf={nf}, depth={depth}, ne={ne}, RMSE_test={test_rmse_pct:.2f}%, MAPE_test={test_mape:.2f}%, time={elapsed/60:.1f}min \n\t\t\tRMSE_train={train_rmse_pct:.2f}%, MAPE_train={train_mape:.2f}%")

# === SAVE RESULTS ===
df_new = pd.DataFrame(results_new)
df_new.to_csv(results_path, sep='\t', mode=write_mode, index=False, header=write_header)
print(f"{'Created' if write_header else 'Appended'} {len(df_new)} rows to {results_path}")

# === SUMMARY REPORT ===
df_all = pd.read_csv(results_path, sep='\t')
print("\n=== SUMMARY REPORT ===")
print("Columns found:", df_all.columns.tolist())
print("n_features values:", sorted(df_all['n_features'].unique()))
print("max_depth values:", sorted(df_all['max_depth'].unique()))

# === TEMPORAL ANALYSIS ===
results = pd.DataFrame({
    'Actual': y_te,
    'Predicted': y_pred_test,
    'Hour': df_test['time'].dt.hour,
    'DayOfWeek': df_test['time'].dt.dayofweek,
    'IsHoliday': df_test.get('is_holiday', pd.Series(False, index=df_test.index))
})

hourly_performance = results.groupby('Hour').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')
dow_performance = results.groupby('DayOfWeek').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')
holiday_performance = results.groupby('IsHoliday').apply(lambda x: (100/np.mean(y_te))*np.sqrt(mean_squared_error(x['Actual'], x['Predicted']))).rename('RMSE')

print("\nPerformance by Hour:")
print(hourly_performance)
print("\nPerformance by Day of Week:")
print(dow_performance)
print("\nPerformance on Holidays:")
print(holiday_performance)

# === SAVE PERFORMANCE PLOTS ===
def save_bar_plot(series, title):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=series.index.astype(str), y=series.values, marker_color='#DA1884'))
    fig.update_layout(title=title, xaxis_title=series.name, yaxis_title='RMSE')
    fig.show()
    # fig.write_image(f"{plot_dir}/{filename}.png")

save_bar_plot(hourly_performance, 'RMSE by Hour of Day')
save_bar_plot(dow_performance, 'RMSE by Weekday')
save_bar_plot(holiday_performance, 'RMSE by Holiday')



In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Load & clean ===
df = pd.read_csv('Results/BR/XGBoost/xgb_evolution_filtered_integer.txt', sep='\t', skipinitialspace=True)
df.columns = df.columns.str.strip()

# === Determine best test RMSE ===
best_row = df.loc[df['rmse_percentual_test'].idxmin()]
best_ne = best_row['n_estimators']
best_md = best_row['max_depth']
print(f"🎯 Best test RMSE: {best_row['rmse_percentual_test']:.2f}% @ "
      f"n_estimators={best_ne}, max_depth={best_md}, n_features={best_row['n_features']}")

# === Prepare grids ===
df_ne = df[df['n_estimators'] == best_ne]
df_md = df[df['max_depth'] == best_md]

nf1, mds = np.sort(df_ne['n_features'].unique()), np.sort(df_ne['max_depth'].unique())
nf2, nes = np.sort(df_md['n_features'].unique()), np.sort(df_md['n_estimators'].unique())

def pivot_vals(subdf, row_vals, col_vals, idx_col, col_col, val_col):
    pt = subdf.pivot_table(index=idx_col, columns=col_col, values=val_col)
    return pt.loc[row_vals, col_vals].values

Z = {
    'rmse_train_md': pivot_vals(df, mds, nf1, 'max_depth', 'n_features', 'rmse_percentual_train'),
    'rmse_test_md':  pivot_vals(df, mds, nf1, 'max_depth', 'n_features', 'rmse_percentual_test'),
    'mape_train_md': pivot_vals(df, mds, nf1, 'max_depth', 'n_features', 'mape_train'),
    'mape_test_md':  pivot_vals(df, mds, nf1, 'max_depth', 'n_features', 'mape_test'),
    'rmse_train_ne': pivot_vals(df, nes, nf2, 'n_estimators', 'n_features', 'rmse_percentual_train'),
    'rmse_test_ne':  pivot_vals(df, nes, nf2, 'n_estimators', 'n_features', 'rmse_percentual_test'),
    'mape_train_ne': pivot_vals(df, nes, nf2, 'n_estimators', 'n_features', 'mape_train'),
    'mape_test_ne':  pivot_vals(df, nes, nf2, 'n_estimators', 'n_features', 'mape_test'),
}

# === build 2×2 plot ===
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'scene'},{'type':'scene'}], [{'type':'scene'},{'type':'scene'}]],
    subplot_titles=[
        "RMSE vs Features × Max Depth",
        "MAPE vs Features × Max Depth",
        "RMSE vs Features × #Estimators",
        "MAPE vs Features × #Estimators"
    ],
    horizontal_spacing=0.1, vertical_spacing=0.1
)

rmse_scale = [[0,'#FFDF00'], [0.5,'#FFEB66'], [1,'#FFF599']]
mape_scale = [[0,'#003D1A'], [0.5,'#007F3D'], [1,'#00BF7F']]

# Add surfaces
fig.add_trace(go.Surface(
    x=nf1, y=mds, z=Z['rmse_test_md'], colorscale=rmse_scale, name='Test RMSE', showscale=False),
    row=1, col=1)
fig.add_trace(go.Surface(
    x=nf1, y=mds, z=Z['rmse_train_md'], colorscale=rmse_scale, opacity=0.6, name='Train RMSE', showscale=False),
    row=1, col=1)

fig.add_trace(go.Surface(
    x=nf1, y=mds, z=Z['mape_test_md'], colorscale=mape_scale, name='Test MAPE', showscale=False),
    row=1, col=2)
fig.add_trace(go.Surface(
    x=nf1, y=mds, z=Z['mape_train_md'], colorscale=mape_scale, opacity=0.6, name='Train MAPE', showscale=False),
    row=1, col=2)

fig.add_trace(go.Surface(
    x=nf2, y=nes, z=Z['rmse_test_ne'], colorscale=rmse_scale, name='Test RMSE', showscale=False),
    row=2, col=1)
fig.add_trace(go.Surface(
    x=nf2, y=nes, z=Z['rmse_train_ne'], colorscale=rmse_scale, opacity=0.6, name='Train RMSE', showscale=False),
    row=2, col=1)

fig.add_trace(go.Surface(
    x=nf2, y=nes, z=Z['mape_test_ne'], colorscale=mape_scale, name='Test MAPE', showscale=False),
    row=2, col=2)
fig.add_trace(go.Surface(
    x=nf2, y=nes, z=Z['mape_train_ne'], colorscale=mape_scale, opacity=0.6, name='Train MAPE', showscale=False),
    row=2, col=2)

# Set axis titles
fig.update_scenes(
    xaxis_title="#Features", yaxis_title="Max Depth", zaxis_title="Error (%)",
    row=1, col=1
)
fig.update_scenes(
    xaxis_title="#Features", yaxis_title="Max Depth", zaxis_title="Error (%)",
    row=1, col=2
)
fig.update_scenes(
    xaxis_title="#Features", yaxis_title="#Estimators", zaxis_title="Error (%)",
    row=2, col=1
)
fig.update_scenes(
    xaxis_title="#Features", yaxis_title="#Estimators", zaxis_title="Error (%)",
    row=2, col=2
)

fig.update_layout(
    title="XGBoost: Train vs Test Errors @ best params",
    width=1400, height=1200, showlegend=False
)

fig.show()


### LSTM

In [ ]:
import numpy as np
import pandas as pd
import os
import time
import itertools
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam

# === CONFIGURATION ===
results_path = 'Results/BR/LSTM/lstm_calendar.txt'
os.makedirs(os.path.dirname(results_path), exist_ok=True)

# === CLEANING FUNCTION ===
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

# === SEQUENCE CREATION ===
def create_sequences(X, y, n_steps):
    X_seq, y_seq = [], []
    for i in range(len(X) - n_steps):
        X_seq.append(X[i:i + n_steps])
        y_seq.append(y[i + n_steps])
    return np.array(X_seq), np.array(y_seq)

# === DATASET PREPARATION ===
exclude_cols = {'time', 'timestamp', 'value'}
features = [c for c in df_train.columns if c not in exclude_cols]

# === LOAD EXISTING RESULTS ===
if os.path.exists(results_path):
    df_existing = pd.read_csv(results_path, sep='\t', skipinitialspace=True)
    df_existing = clean_columns(df_existing)
    write_header = False
    write_mode = 'a'
else:
    df_existing = pd.DataFrame(columns=['n_features','layers','epochs','batch_size','lr','rmse_pct','mape','train_rmse_pct','train_mape','more_than_10_min'])
    write_header = True
    write_mode = 'w'

existing_tuples = set(
    (row['n_features'], tuple(eval(row['layers'])), row['epochs'], row['batch_size'], row['lr'])
    for _, row in df_existing.iterrows()
)

# === HYPERPARAMETER GRID ===
layer_configs = [[64, 16], [64, 64], [128, 64], [32, 32, 8], [32, 32, 32], [32, 32, 32, 16], [64, 64, 64, 16]]  # reduce for testing
epochs_list = [5, 10, 15]  # reduce for testing
batch_sizes = [256, 512, 1024]  # reduce for testing
learning_rates = [0.005, 0.001, 0.0005, 0.0001]  # reduce for testing
n_steps = 24

# === FEATURE VARIATION LOOP ===
results_new = []
for nf in range(1, len(features) + 1):
    subset = features[:nf]

    X_tr = df_train[subset].to_numpy()
    X_te = df_test[subset].to_numpy()
    y_tr = df_train['value'].to_numpy().reshape(-1, 1)
    y_te = df_test['value'].to_numpy().reshape(-1, 1)

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    X_tr_scaled = scaler_X.fit_transform(X_tr)
    X_te_scaled = scaler_X.transform(X_te)
    y_tr_scaled = scaler_y.fit_transform(y_tr)
    y_te_scaled = scaler_y.transform(y_te)

    X_tr_seq, y_tr_seq = create_sequences(X_tr_scaled, y_tr_scaled, n_steps)
    X_te_seq, y_te_seq = create_sequences(X_te_scaled, y_te_scaled, n_steps)

    X_tr_seq = X_tr_seq.reshape((X_tr_seq.shape[0], n_steps, nf))
    X_te_seq = X_te_seq.reshape((X_te_seq.shape[0], n_steps, nf))

    for layers, epochs, batch_size, lr in itertools.product(layer_configs, epochs_list, batch_sizes, learning_rates):
        config_key = (nf, tuple(layers), epochs, batch_size, lr)
        if config_key in existing_tuples:
            continue

        print(f"Training LSTM (n_features={nf}): layers={layers}, epochs={epochs}, batch_size={batch_size}, lr={lr}")
        t0 = time.perf_counter()

        model = Sequential()
        model.add(Input(shape=(n_steps, nf)))
        for i, units in enumerate(layers):
            return_seq = (i < len(layers) - 1)
            model.add(LSTM(units, return_sequences=return_seq))
        model.add(Dense(1))

        model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
        model.fit(X_tr_seq, y_tr_seq, epochs=epochs, batch_size=batch_size, verbose=0)

        elapsed = time.perf_counter() - t0

        y_pred = scaler_y.inverse_transform(model.predict(X_te_seq, verbose=0)).flatten()
        y_train_pred = scaler_y.inverse_transform(model.predict(X_tr_seq, verbose=0)).flatten()
        y_te_seq_inv = scaler_y.inverse_transform(y_te_seq).flatten()
        y_tr_seq_inv = scaler_y.inverse_transform(y_tr_seq).flatten()

        rmse_pct = 100 * np.sqrt(mean_squared_error(y_te_seq_inv, y_pred)) / np.mean(y_te_seq_inv)
        mape = 100 * np.mean(np.abs((y_te_seq_inv - y_pred) / y_te_seq_inv))
        train_rmse_pct = 100 * np.sqrt(mean_squared_error(y_tr_seq_inv, y_train_pred)) / np.mean(y_tr_seq_inv)
        train_mape = 100 * np.mean(np.abs((y_tr_seq_inv - y_train_pred) / y_tr_seq_inv))

        results_new.append({
            'n_features': nf,
            'layers': str(layers),
            'epochs': epochs,
            'batch_size': batch_size,
            'lr': lr,
            'rmse_pct': rmse_pct,
            'mape': mape,
            'train_rmse_pct': train_rmse_pct,
            'train_mape': train_mape,
            'more_than_10_min': elapsed / 60
        })

        print(f"Test RMSE={rmse_pct:.2f}%, Test MAPE={mape:.2f}% | \nTrain RMSE={train_rmse_pct:.2f}%, Train MAPE={train_mape:.2f}% | \nTime={elapsed/60:.1f}min\n")

# === SAVE RESULTS ===
df_new = pd.DataFrame(results_new)
df_new.to_csv(results_path, sep='\t', mode=write_mode, index=False, header=write_header)
print(f"{'Created' if write_header else 'Appended'} {len(df_new)} rows to {results_path}")


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Load & clean ===
df = pd.read_csv('Results/BR/LSTM/lstm_calendar.txt', sep='\t', skipinitialspace=True)
df.columns = df.columns.str.strip()

# === Convert layers from string to tuple ===
df['layers'] = df['layers'].apply(lambda x: tuple(eval(x)))

# === Determine best RMSE ===
best_row = df.loc[df['rmse_pct'].idxmin()]
print(f"\U0001F3AF Best RMSE: {best_row['rmse_pct']:.2f}% @ n_features={best_row['n_features']}, layers={best_row['layers']}, "
      f"epochs={best_row['epochs']}, batch_size={best_row['batch_size']}, lr={best_row['lr']}")

# === Utilities ===
# def pivot_vals(subdf, row_vals, col_vals, idx_col, col_col, val_col):
#     pt = subdf.pivot_table(index=idx_col, columns=col_col, values=val_col)
#     return pt.loc[row_vals, col_vals].values

def pivot_vals(subdf, row_vals, col_vals, idx_col, col_col, val_col, best_row=None, log_arrays=True):
    subdf[idx_col] = subdf[idx_col].apply(lambda x: str(x) if isinstance(x, (tuple, list)) else x)
    subdf[col_col] = subdf[col_col].apply(lambda x: str(x) if isinstance(x, (tuple, list)) else x)
    pt = subdf.pivot_table(index=idx_col, columns=col_col, values=val_col)

    def to_array(x):
        if isinstance(x, str):
            return np.array([float(i) for i in x.replace('(', '').replace(')', '').split(',')])
        elif isinstance(x, (tuple, list, np.ndarray)):
            return np.array(x, dtype=float)
        else:
            return np.array([float(x)])

    def as_key(x):
        """Converts tuple/list to string key for pivot access, if needed."""
        return str(x) if isinstance(x, (tuple, list, np.ndarray)) else x

    if log_arrays:
        print("\n--- Parsed row_vals ---")
        for x in row_vals:
            try:
                arr = to_array(x)
                print(f"{x!r} -> {arr} (shape={arr.shape})")
            except Exception as e:
                print(f"{x!r} -> Error: {e}")

        print("\n--- Parsed col_vals ---")
        for x in col_vals:
            try:
                arr = to_array(x)
                print(f"{x!r} -> {arr} (shape={arr.shape})")
            except Exception as e:
                print(f"{x!r} -> Error: {e}")

    if best_row is not None:
        row_target = to_array(best_row[idx_col])
        col_target = to_array(best_row[col_col])
        row_len = len(row_target)
        col_len = len(col_target)

        row_vals = [x for x in row_vals if len(to_array(x)) == row_len]
        col_vals = [x for x in col_vals if len(to_array(x)) == col_len]

        row_vals = sorted(row_vals, key=lambda x: float(np.linalg.norm(to_array(x) - row_target)))
        col_vals = sorted(col_vals, key=lambda x: float(np.linalg.norm(to_array(x) - col_target)))

    # Convert all access keys to strings (to match the index/column labels)
    row_keys = [as_key(x) for x in row_vals]
    col_keys = [as_key(x) for x in col_vals]

    # Final safe access
    try:
        result = pt.loc[row_keys, col_keys].values
    except KeyError as e:
        print("\n[ERROR] KeyError accessing pivot table:")
        print(f"Row keys: {row_keys}")
        print(f"Column keys: {col_keys}")
        print(f"Pivot index: {list(pt.index)}")
        print(f"Pivot columns: {list(pt.columns)}")
        raise e

    return result



# === Unique values ===
n_features = np.sort(df['n_features'].unique())
layers = sorted(df['layers'].unique(), key=lambda x: (len(x), x))
layer_labels = [str(l) for l in layers]
epochs = np.sort(df['epochs'].unique())
batch_sizes = np.sort(df['batch_size'].unique())
lrs = np.sort(df['lr'].unique())

# === Prepare pivot data ===
Z = {}
for metric in ['rmse_pct', 'mape']:
    for hp, values in zip(['layers', 'epochs', 'batch_size', 'lr'], [layers, epochs, batch_sizes, lrs]):
        key_test = f"{metric}_test_{hp}"
        key_train = f"{metric}_train_{hp}"

        Z[key_test] = pivot_vals(df, values, n_features, hp, 'n_features', metric, best_row=best_row)
        Z[key_train] = pivot_vals(df, values, n_features, hp, 'n_features', 'train_' + metric, best_row=best_row)

# === Plotting ===
fig = make_subplots(
    rows=4, cols=2,
    specs=[[{'type':'scene'}]*2]*4,
    subplot_titles=[
        "1.1 RMSE vs Features × Layers", "1.2 MAPE vs Features × Layers",
        "2.1 RMSE vs Features × Epochs", "2.2 MAPE vs Features × Epochs",
        "3.1 RMSE vs Features × Batch Size", "3.2 MAPE vs Features × Batch Size",
        "4.1 RMSE vs Features × Learning Rate", "4.2 MAPE vs Features × Learning Rate"
    ],
    horizontal_spacing=0.1, vertical_spacing=0.1
)

# Axis values
axes = {
    'layers': layers,
    'epochs': epochs,
    'batch_size': batch_sizes,
    'lr': lrs
}

# Plot surfaces
color_scales = {
    'rmse_pct': [[0,'#FFDF00'], [0.5,'#FFEB66'], [1,'#FFF599']],
    'mape': [[0,'#003D1A'], [0.5,'#007F3D'], [1,'#00BF7F']]
}

row_map = {'layers': 1, 'epochs': 2, 'batch_size': 3, 'lr': 4}

for i, metric in enumerate(['rmse_pct', 'mape']):
    for j, hp in enumerate(['layers', 'epochs', 'batch_size', 'lr']):
        row, col = row_map[hp], i + 1
        y_vals = axes[hp]

        # Convert layer tuples to string labels for the plot
        y_axis_vals = list(range(len(y_vals))) if hp == 'layers' else y_vals
        z_test = Z[f"{metric}_test_{hp}"]
        z_train = Z[f"{metric}_train_{hp}"]

        fig.add_trace(go.Surface(
            x=n_features, y=y_axis_vals, z=z_test,
            colorscale=color_scales[metric], name='Test', showscale=False),
            row=row, col=col)

        fig.add_trace(go.Surface(
            x=n_features, y=y_axis_vals, z=z_train,
            colorscale=color_scales[metric], opacity=0.6, name='Train', showscale=False),
            row=row, col=col)

        # Set layer tick labels as string versions of the layer tuple
        if hp == 'layers':
            fig.update_scenes(
                yaxis=dict(
                    tickmode='array',
                    tickvals=list(range(len(layer_labels))),
                    ticktext=layer_labels
                ),
                row=row, col=col
            )

# Update layout
for row in range(1, 5):
    for col in range(1, 3):
        hp = list(row_map.keys())[row-1]
        fig.update_scenes(
            xaxis_title="#Features", yaxis_title=hp.capitalize(), zaxis_title="Error (%)",
            row=row, col=col
        )

fig.update_layout(
    title="LSTM Error Surface: Train vs Test",
    width=1400, height=1800, showlegend=False
)

fig.show()

***Be carefull before running all these following cells***

# Test for better parameters 

In [ ]:
import numpy as np
from numpy.typing import NDArray
import pandas as pd
import time
from math import ceil
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import multiprocessing as mp
import warnings
import os


def stairway_to_heaven_signal(df: pd.DataFrame, keep_freqs: np.ndarray, keep_mags: np.ndarray) -> pd.DataFrame:
    """
    Generate stair-like signals repeating within each cycle defined by the FFT filtered frequencies.
    Each signal will increase from 1 up to the period (in time steps), then restart from 1 again.

    Returns a DataFrame with:
    - 'value' column (original values)
    - 'fft_{period}h_signal' columns for each frequency
    - Also returns a single interactive plot showing all signals
    """
    df_out = pd.DataFrame(index=df.index)
    df_out = df

    N = len(df)
    t = np.arange(N)  # sample index as time base (assuming uniform sampling)

    for f, mag in zip(keep_freqs, keep_mags):
        period = int(round(1 / f))
        if period < 1:
            continue  # skip invalid periods

        signal = (t % period) + 1

        colname = f"fft_{period}h_signal"
        df_out[colname] = signal

    return df_out

def filter_similar_periods(periods, mags, th_hours, th_week, th_months, th_years):
    drop = set()
    def domain(idxs, convert_fn, thresh, unit):
        for i in range(len(idxs)):
            for j in range(i+1, len(idxs)):
                a, b = idxs[i], idxs[j]
                da, db = convert_fn(periods[a]), convert_fn(periods[b])
                if abs(da - db) <= thresh:
                    drop.add(a if mags[a] < mags[b] else b)
                    warnings.warn(f"Dropping {unit}-close: {periods[a]:.1f}h vs {periods[b]:.1f}h")

    hrs = [i for i,p in enumerate(periods) if p < 24*7*1.1]
    wks = [i for i,p in enumerate(periods) if 24*7*0.9 <= p < 24*30*1.1]
    mos = [i for i,p in enumerate(periods) if 24*30*0.9 <= p < 24*365*1.1]
    yrs = [i for i,p in enumerate(periods) if p >= 24*365*0.9]
    domain(hrs, lambda x: x, th_hours, 'hour')
    domain(wks, lambda x: x/24/7, th_week, 'week')
    domain(mos, lambda x: x/24/30, th_months, 'month')
    domain(yrs, lambda x: x/24/365, th_years, 'year')
    return [i for i in range(len(periods)) if i not in drop]

def fft_light_without_plot(df: pd.DataFrame, n_peaks, trunc_factor, th_hours, th_week, th_months, th_years):
    x = df['value'].values
    N = len(x)
    fftc = np.fft.fft(x)
    freqs = np.fft.fftfreq(N, d=1.0)
    mask = freqs > 0
    fpos = freqs[mask]
    mpos = np.abs(fftc[mask]) / (N / 2)

    # Truncate
    f_plot = fpos[:len(fpos)//trunc_factor]
    m_plot = mpos[:len(mpos)//trunc_factor]
    p_plot = 1 / f_plot

    # Top N peaks
    top_idxs = np.argpartition(m_plot, -n_peaks)[-n_peaks:]
    top_idxs = top_idxs[np.argsort(m_plot[top_idxs])[::-1]]
    peak_freqs = f_plot[top_idxs]
    peak_mags = m_plot[top_idxs]
    peak_periods = 1 / peak_freqs

    keep = filter_similar_periods(periods = peak_periods, 
                                  mags = peak_mags, 
                                  th_hours = th_hours, 
                                  th_week = th_week, 
                                  th_months = th_months, 
                                  th_years = th_years)
    keep_freqs = peak_freqs[keep]
    keep_mags = peak_mags[keep]

    df_full = stairway_to_heaven_signal(df = df,
                                        keep_freqs = keep_freqs,
                                        keep_mags = keep_mags)
    
    return df_full

def fft_features_with_params(df, n_components, truncation_factor,
                              th_hours, th_week, th_months, th_years):
    
    df_full = fft_light_without_plot(df, n_components, truncation_factor, th_hours, th_week, th_months, th_years)
    df_train, df_test = train_test_split(df_full)
    return df_train, df_test

def grid_search_fft_rf(
    df,
    n_components: int,
    period_ticks: NDArray[np.float64],
    truncation_factor: int,
    max_depths: list,
    n_estimators: list,
    feature_steps: list,
    fft_thresholds: list,
    simulation_id=0
):
    records = []

    fft_dfs = {}
    for th in fft_thresholds:
        key = (th['th_hours'], th['th_week'], th['th_months'], th['th_years'])
        # EXPLICITLY unpack each threshold parameter instead of using **th
        df_tr, df_te = fft_features_with_params(
            df,
            n_components=n_components,
            truncation_factor=truncation_factor,
            th_hours=th['th_hours'],
            th_week=th['th_week'],
            th_months=th['th_months'],
            th_years=th['th_years']
        )
        fft_dfs[key] = (df_tr, df_te)

    for key, (df_tr, df_te) in fft_dfs.items():
        exclude_cols = {'time', 'timestamp', 'value'}
        features = [c for c in df_tr.columns if c not in exclude_cols]

        for n_feats in feature_steps:
            X_tr = df_tr[features].values
            X_te = df_te[features].values
            y_tr = df_tr['value'].values.ravel()
            y_te = df_te['value'].values.ravel()

            if X_tr.shape[0] != y_tr.shape[0]:
                print(f"Skipping invalid training pair: X={X_tr.shape}, y={y_tr.shape}")
                continue

            for d, n_e in zip(max_depths, n_estimators):
                t0 = time.perf_counter()
                model = xgb.XGBRegressor(
                    max_depth=d,
                    n_estimators=n_e,
                    n_jobs=mp.cpu_count() // 2,
                    tree_method="hist",
                    verbosity=0
                )
                model.fit(X_tr, y_tr)
                elapsed = time.perf_counter() - t0

                y_pred = model.predict(X_te)
                rmse = 100 * np.sqrt(mean_squared_error(y_te, y_pred)) / np.mean(y_te)
                mape = 100 * np.mean(np.abs((y_te - y_pred) / y_te))

                records.append({
                    'simulation_id': simulation_id,
                    'th_hours': key[0], 'th_week': key[1],
                    'th_months': key[2], 'th_years': key[3],
                    'n_fft_feats': n_feats,
                    'max_depth': d,
                    'n_estimators': n_e,
                    'rmse_pct': rmse,
                    'mape_pct': mape,
                    'time_s': elapsed
                })

                print(f"[Sim {simulation_id}] TH={key}, feats={n_feats}, depth={d}, estimators={n_e}: RMSE={rmse:.2f}%, MAPE={mape:.2f}%")

    return pd.DataFrame(records)

if __name__ == "__main__":
    np.random.seed(14)
    n_simulations = 30
    all_results = []

    for sim in range(n_simulations):
        fft_thresholds = [{
            'th_hours': round(np.random.uniform(1.0, 23.0), 2),
            'th_week': round(np.random.uniform(1.0, 6.0), 2),
            'th_months': round(np.random.uniform(1.0, 11.0), 2),
            'th_years': round(np.random.uniform(1.0, 5.0), 2),
        } for _ in range(15)]  # try 3 FFT threshold sets per simulation

        max_depths = [15]
        n_estimators = [450]
        M = 450  # number of FFT components
        k = 1    # number of feature subsets to evaluate

        feature_steps = sorted(np.random.choice(np.arange(1, M + 1), size=k, replace=False))

        results_df = grid_search_fft_rf(
            df,
            n_components=M,
            period_ticks=np.array([24 * 365, 24 * 7, 24, 12, 8]),
            truncation_factor=5,
            max_depths=max_depths,
            n_estimators=n_estimators,
            feature_steps=feature_steps,
            fft_thresholds=fft_thresholds,
            simulation_id=sim
        )

        all_results.append(results_df)

    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_csv("fft_rf_multi_simulation_results_BR.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load your FFT RF simulation results
file_path = "fft_rf_multi_simulation_results_BR.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())
assert 'rmse_pct' in df.columns and 'mape_pct' in df.columns

# Quick error summary
print("\nRMSE % summary:\n", df['rmse_pct'].describe())
print("\nMAPE % summary:\n", df['mape_pct'].describe())

# Pairwise relationship plot
sns.pairplot(
    df,
    vars=['rmse_pct','mape_pct','n_fft_feats','th_hours','th_week','th_months','th_years'],
    kind='reg',
    plot_kws={'line_kws':{'color':'red','alpha':0.3}}
)
plt.suptitle("Pairwise relationships (with regression fit)", y=1.02)
plt.show()

# Global modes for slicing
top50 = df.nsmallest(50, 'rmse_pct')
modes = {
    'th_week':   top50['th_week'].mode()[0],
    'th_months': top50['th_months'].mode()[0],
    'th_years':  top50['th_years'].mode()[0]
}
print("\nUsing fixed thresholds:", modes)

# Filter slice by modal thresholds
slice_df = df[
    (df['th_week']   == modes['th_week']) &
    (df['th_months'] == modes['th_months']) &
    (df['th_years']  == modes['th_years'])
]

# Unique threshold combinations for heatmap generation
unique_combos = (
    df
    .groupby(['th_week','th_months','th_years'])
    .size()
    .reset_index(name='count')
    .query('count >= 5')
)

# Generate heatmaps per valid threshold combo
for _, row in unique_combos.iterrows():
    w, m, y = row['th_week'], row['th_months'], row['th_years']
    combo_df = df[
        (df['th_week'] == w) &
        (df['th_months'] == m) &
        (df['th_years'] == y)
    ]
    pivot = combo_df.pivot_table(
        index='th_hours', columns='n_fft_feats', values='rmse_pct', aggfunc='mean'
    )
    if pivot.size == 0:
        continue
    plt.figure(figsize=(6,4))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap='viridis')
    plt.title(f"RMSE % (th_week={w}, th_months={m}, th_years={y})\n{len(combo_df)} runs")
    plt.ylabel('th_hours')
    plt.xlabel('# FFT features')
    plt.tight_layout()
    plt.show()

# Summary heatmap for modal combo
pivot_rmse = slice_df.pivot_table(
    index='th_hours', columns='n_fft_feats', values='rmse_pct', aggfunc='mean'
)
if pivot_rmse.size > 0:
    plt.figure(figsize=(8,5))
    sns.heatmap(pivot_rmse, annot=True, fmt=".1f", cmap='viridis')
    plt.title(
        f"Mean RMSE %\n"
        f"(th_week={modes['th_week']}, th_months={modes['th_months']}, th_years={modes['th_years']})"
    )
    plt.ylabel('th_hours')
    plt.xlabel('# FFT features')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data for modal threshold combination.")

# Boxplots for RMSE and MAPE
fig, axes = plt.subplots(1,3, figsize=(15,4))
sns.boxplot(x='th_hours', y='rmse_pct', data=df, ax=axes[0])
axes[0].set_title("RMSE % by th_hours")
sns.boxplot(x='n_fft_feats', y='rmse_pct', data=df, ax=axes[1])
axes[1].set_title("RMSE % by # FFT features")
sns.boxplot(x='max_depth', y='rmse_pct', data=df, ax=axes[2])
axes[2].set_title("RMSE % by RF max_depth")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1,3, figsize=(15,4))
sns.boxplot(x='th_hours', y='mape_pct', data=df, ax=axes[0])
axes[0].set_title("MAPE % by th_hours")
sns.boxplot(x='n_fft_feats', y='mape_pct', data=df, ax=axes[1])
axes[1].set_title("MAPE % by # FFT features")
sns.boxplot(x='max_depth', y='mape_pct', data=df, ax=axes[2])
axes[2].set_title("MAPE % by RF max_depth")
plt.tight_layout()
plt.show()

# Best and worst configurations
print("\nTop 10 configs by RMSE:")
print(df.nsmallest(10, 'rmse_pct')[[
    'th_hours','th_week','th_months','th_years',
    'n_fft_feats','max_depth','n_estimators','rmse_pct'
]])

print("\nWorst 10 configs by RMSE:")
print(df.nlargest(10, 'rmse_pct')[[
    'th_hours','th_week','th_months','th_years',
    'n_fft_feats','max_depth','n_estimators','rmse_pct'
]])
